# Query-Aware Flow Diffusion for Graph-Based RAG (QAFD-RAG)
### End-to-End Interactive Demonstration

This notebook demonstrates the complete QAFD-RAG pipeline for multi-hop question answering:
1. **Dataset Loading**: Multi-hop QA sample inspection (MuSiQue, HotpotQA, 2WikiMultiHopQA)
2. **Heterogeneous Graph Construction**: Passage nodes, entity nodes, fact triples, associations, and synonymy edges
3. **Query-Aware Flow Diffusion**: Push-relabel propagation with $\alpha=2.0$, $\epsilon=0.01$, $\text{step\_size}=0.2$
4. **Multi-Hop Path Tracing**: Path extraction along maximum flow channels
5. **LLM Generation & Evaluation**: Strict grounded answer generation and evaluation (EM, F1, MRR)

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

from data.loaders import MuSiQueLoader
from graph import PassageEntityGraph, EntityGraph
from embeddings import get_embedding_model
from retrieval import QAFDFlowDiffusionRetriever
from models import LLMClient
from evaluation import compute_qa_metrics, compute_retrieval_metrics

print("QAFD-RAG modules successfully loaded.")

## 1. Load Multi-Hop QA Sample
Let us inspect a 2-hop question from MuSiQue requiring reasoning across Alexander Graham Bell and Scotland.

In [ ]:
loader = MuSiQueLoader("../data/raw/musique/sample.jsonl")
items = loader.load()
sample = items[0]

print(f"Question ID: {sample.id}")
print(f"Question:    {sample.question}")
print(f"Gold Answer: {sample.answer}")
print(f"Aliases:     {sample.answer_aliases}")
print(f"Passages:    {len(sample.passages)} (Supporting: {len(sample.get_gold_passage_ids())})")

## 2. Construct Heterogeneous Passage-Entity Graph
We construct the heterogeneous graph containing passage nodes, extracted entities, fact triples, and synonymy edges.

In [ ]:
embedding_model = get_embedding_model("local-hash-embed", dimension=128)
peg = PassageEntityGraph(embedding_model=embedding_model, synonymy_threshold=0.80)
peg.build_from_passages(sample.passages)

print(f"Total Nodes:    {peg.num_nodes} (Passages: {peg.num_passages}, Entities: {peg.num_entities})")
print(f"Total Edges:    {peg.num_edges}")

## 3. Query-Aware Flow Diffusion (Push-Relabel Propagation)
We initialize the potential gradient field $h(u) = \alpha \cdot s_0(u)$ and excess flow $e(u)$, then execute push and relabel operations.

In [ ]:
p_map = {p.id: p for p in sample.passages}
retriever = QAFDFlowDiffusionRetriever(
    graph=peg.graph,
    passages=p_map,
    embedding_model=embedding_model,
    alpha=2.0,
    epsilon=0.01,
    step_size=0.2
)

ret_res = retriever.retrieve(sample.question, top_k=2)

print(f"Diffusion Steps:  {ret_res.diffusion_steps}")
print(f"Retrieval Latency: {ret_res.latency_ms:.2f} ms")
print("\nRetrieved Top-K Passages:")
for i, p in enumerate(ret_res.retrieved_passages, 1):
    print(f"  {i}. {p.title} (Supporting: {p.is_supporting})")

## 4. Traced Multi-Hop Reasoning Path
Inspect the flow path discovered by the push-relabel propagation:

In [ ]:
for path in ret_res.paths:
    print(path)

## 5. Answer Generation & Evaluation
Send the retrieved multi-hop context to the LLM and evaluate with Exact Match and F1.

In [ ]:
llm = LLMClient(model_name="local-extractive")
context = "\n\n".join([f"[{p.title}] {p.text}" for p in ret_res.retrieved_passages])
prediction = llm.answer_question(sample.question, context)

qa_metrics = compute_qa_metrics(prediction, sample.answer, sample.answer_aliases)
ret_metrics = compute_retrieval_metrics([p.id for p in ret_res.retrieved_passages], sample.get_gold_passage_ids())

print(f"Predicted Answer: {prediction}")
print(f"Gold Answer:      {sample.answer}")
print(f"Exact Match:      {qa_metrics['exact_match']:.2f}")
print(f"F1 Score:         {qa_metrics['f1']:.2f}")
print(f"MRR:              {ret_metrics['mrr']:.2f}")
print(f"Recall@2:         {ret_metrics['recall@2']:.2f}")